#### pip install litellm

In [2]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.metrics.collections import AnswerRelevancy
import os


# -------------------------
# LLM: OpenRouter
# -------------------------
client = AsyncOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

llm = llm_factory(
    "openai/gpt-oss-120b",
    client=client,
)


# -------------------------
# Embeddings: Ollama
# -------------------------

embeddings = embedding_factory(
    "litellm",
    model="ollama/qwen3-embedding:4b",
    base_url="http://localhost:11434",
)
# embeddings = embedding_factory("openai", model="text-embedding-3-small", client=client)

scorer = AnswerRelevancy(llm=llm, embeddings=embeddings)

ImportError: Failed to import litellm. Please install the required package: pip install litellm

In [ ]:
# Example 1 : Response answers the question but drifts into tangential information
# Mentioning the compromise between Sydney and Melbourne dilutes relevancy
result = await scorer.ascore(
    user_input="What is the capital of Australia?",
    response="Australia is a large country in the Southern Hemisphere. It has many major cities including Sydney, Melbourne, and Brisbane. Canberra is the capital city, chosen as a compromise between Sydney and Melbourne. Australia also has a diverse economy driven by mining and agriculture."
)
print(f"Response Relevancy Score: {result.value}")

Response Relevancy Score: 0.9621900768990267


In [ ]:
# Example 2: Response is direct and precisely answers the question with no filler
result = await scorer.ascore(
    user_input="When was the first Super Bowl played?",
    response="The first Super Bowl was played on January 15, 1967."
)
print(f"Response Relevancy Score: {result.value}")

Response Relevancy Score: 1.0000000000000002


In [ ]:
# Example 3 : Response talks about water as a topic but never states its boiling point
result = await scorer.ascore(
    user_input="What is the boiling point of water?",
    response="Water is a fascinating substance found all over the Earth. It is essential for all known forms of life and covers about 71 percent of the Earth's surface. Water is found in oceans, rivers, lakes, and glaciers and plays a key role in regulating climate."
)
print(f"Response Relevancy Score: {result.value}")

Response Relevancy Score: 0.26754335611869756
